# SPOTPY calibration steps

**Code under development**
Note: sampler needs set-up and debugging

## Setup

1. Load raven files to Data/Calibration/{model name}


In [ ]:
import os
import sys
import pandas as pd
import spotpy as sp

# Ensure the project directory is in the Python path
prj_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if prj_path not in sys.path:
    sys.path.insert(0, prj_path)
import src.py.raven_py as rv
from src.py.proj_conf import *
from src.py.setup_spotpy import *

import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# DATA_CAL = "../Data/Calibration/cypress"
# Check for Raven.exe in the DATA_CAL folder
raven_exe_path = os.path.join(DATA_CAL, "Raven.exe")
raven_name = "Cypress_New"
# raven_name = "CypressRiverT_ERA5"

In [ ]:
# Set calibration directory
DATA_CAL_NEW = os.path.join(prj_path, r"Data\Calibration", raven_name).replace('\\','/')

# Specify the calibration results file name and location (no .csv extension)
cal_data_path = DATA_CAL + "../Results/fast_rdrs_toffset_20251002_21"

In [ ]:
# Specify necessary functions

def generate_param_tuples(spotpy_results, n_params = 7):
    # Generate tuples for parameter pairwise plots
    parameters = spotpy_results.dtype.names[1:n_params + 1]
    parameter_tuples = [(parameters[i], parameters[j]) for i in range(len(parameters)) for j in range(i + 1, len(parameters))]
    return parameter_tuples

def plot_param_dist(_data, _param, subset_name):
    # Plot distributions of calibration parameters
    plt.hist(_data[_param], bins=10, alpha=0.7, label=f'count {_param}')
    plt.legend()
    plt.title(f'Distribution of {_param} in top 500 of the {subset_name} distribution')
    median_col = np.round(np.median(_data[_param]), 4)
    mean_col = np.round(np.mean(_data[_param]), 4)
    plt.xlabel(f'median: {median_col}, mean: {mean_col}')
    plt.show()

def sensitivity_plot(_data, x_str, y_str, objective_str, filename = None):
    # Plot objective function values (color) against 2 calibration parameters
    
    # Extract the first 3 columns from _data
    x = _data[x_str]
    y = _data[y_str]
    objective = _data[objective_str]

    # Normalize objective for color mapping
    norm_objective = (objective - objective.min()) / (objective.max() - objective.min())

    # Create a color map: blue for low objective, red for high objective
    colors = plt.cm.coolwarm(norm_objective)

    plt.figure(figsize=(8,6))
    scatter = plt.scatter(x, y, c=colors, s=40, edgecolor='k')
    plt.xlabel(x_str)
    plt.ylabel(y_str)
    plt.title(f'FAST Results: {x_str} vs {y_str} colored by {objective_str}')

    # Add colorbar to show objective scale
    sm = plt.cm.ScalarMappable(cmap=plt.cm.coolwarm, norm=plt.Normalize(vmin=objective.min(), vmax=objective.max()))
    sm.set_array([])
    # cbar = plt.colorbar(sm)
    # cbar.set_label('objective')

    if filename:
        plt.savefig(filename)
        print(f"Plot saved as {filename}")

    plt.show()

## Raven test run

Test the Raven script in the main calibration path

In [ ]:
## Checking end of nc timeseries for RDRS

# from netCDF4 import Dataset

# nc_file = Dataset(DATA_CAL+"input/rdrs_runoff_wlat" + ".nc", "r")

# nc_file.variables["time"][:].min() / 8766

# nc_file.variables["time"][:].max() / 8766


In [ ]:
# raven_name = input("Enter the Raven model name (if different):")
if os.path.isfile(raven_exe_path):
    test_confirm = input(f"Raven.exe found at {raven_exe_path}. Do you want to run it? (yes or y to confirm): ")
    if (test_confirm.lower() != 'yes' and test_confirm.lower() != 'y'):
        print("Exiting without running Raven.exe.")
    else:
        print(f"Testing Raven.exe at {raven_exe_path}...")
        # Set working directory to DATA_CAL and run Raven.exe
        rv.run_raven(raven_name, DATA_CAL)
else:
    print(f"Raven.exe not found in location: {DATA_CAL}")

## Calibration set-up and test run

Test Calibration set-up (without mpi parallelization)

In [ ]:

calibrator = spot_setup_rvn(raven_name, DATA_CAL_NEW, obj_fn= "nse_pbias", split_areas = "none", \
                            spring_eval= True, low_seep=False, routing=True, dep_scaling=True, runoff_scaling=True, channel_scaling = True,
                            mpi_cores=0)


In [ ]:
# Use the Pareto Archived Dynamically Dimensioned Search (PADDS) algorithm for calibration
# sampler=sp.algorithms.padds(spot_setup=calibrator, # Core model setup object
#                             dbname = "padds_raven", # Save results of cal run to this database
#                             dbformat='csv')

 
sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
                           parallel='seq',
                           dbname='test',
                           dbformat='csv')

# sampler=sp.algorithms.mc(calibrator, # must use this or ROPE flattens the parameter space and collapses!!
#                            parallel='seq',
#                            dbname='mc_raven',
#                            dbformat='csv')



In [ ]:
rep = 3
sampler.sample(rep)

## REMINDER - try to reduce the evaluation period to 2003 onward to improve performance

# setup = calibrator  # your spot_setup object

# params = [p.optguess for p in setup.parameters]
# print("params:", params)

# sim = setup.simulation(params)
# evals = setup.evaluation()
# obj = setup.objectivefunction(sim, evals)
# print("Objective output:", obj)

## FAST Analysis

Run fast method in mpiexec

With 5 parameters, the number of reps required for FAST is 2245

In [ ]:
# Load FAST results and plot sensitivity

fast_results = sp.analyser.load_csv_results(cal_data_path)
sp.analyser.plot_fast_sensitivity(fast_results, number_of_sensitiv_pars=3)

In [ ]:
fast_results = fast_results[['like1', 'pardep_fact', 'parseep_k', 'part_peak_fact']]

parameter_tuples = generate_param_tuples(fast_results, n_params = 7)

for param1, param2 in parameter_tuples:
    sensitivity_plot(fast_results, param1, param2, 'like1') # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">

## Calibration Analysis

### Define calibration subset

In [ ]:
calibration_results = sp.analyser.load_csv_results(cal_data_path)

In [ ]:
# Only relevant for a converged calibration
# Compare the prior (first 500) and posterior (last 500) distributions of the Spotpy results

calibration_prior = calibration_results[:500]
calibration_post = calibration_results[-500:]

plot_param_dist(calibration_prior, 'like1', 'prior')
plot_param_dist(calibration_post, 'like1', 'post')

In [ ]:
# Some options for selecting the best results for analysis

# calibration_post_best = pd.DataFrame(calibration_results[7500:]).sort_values(by='like1', ascending=False).head(500)

# calibration_post_best = pd.DataFrame(calibration_post).sort_values(by='like1', ascending=False).head(100)

calibration_post_best = pd.DataFrame(calibration_results).sort_values(by='like1', ascending=False).head(200)

### Evaluate prior or post subsets

In [ ]:
# Rename the objective function column for clarity
obj_fn_str = "nse-1-norm(pbias)"
calibration_post_best.rename(columns = {"like1" : obj_fn_str}, inplace=True)

# Plot spotpy calibration results for the selected best results
for col in calibration_post_best.columns:
    plt.hist(calibration_post_best[col], bins=10, alpha=0.7, label=f'count {col}')
    plt.legend()
    plt.title(f'Distribution of {col} in top 500 of the final 2500 ROPE simulations')
    median_col = np.round(calibration_post_best[col].median(), 4)
    mean_col = np.round(calibration_post_best[col].mean(), 4)
    plt.xlabel(f'median: {median_col}, mean: {mean_col}')
    # plt.savefig(f"post_{col}.jpg")
    plt.show()
    

In [ ]:
# View correlations between parameters in the best results
for param1, param2 in parameter_tuples:
        pearson_corr = calibration_post_best[[param1, param2]].corr().iloc[0, 1]
        if np.abs(pearson_corr) > 0.3:
            print(f'Parameter 1: {param1}, Parameter 2: {param2}, corr:{pearson_corr}')

In [ ]:
# Generate heat bi-plots of best results
for param1, param2 in parameter_tuples:
    sensitivity_plot(calibration_post_best, param1, param2, 'nse-1-norm(pbias)') # to save, add <, filename=f"fast_sensitivity_{param1}_{param2}.png">